# PHÂN TÍCH ĐÁNH GIÁ KHÁCH HÀNG E-COMMERCE — EDA & TEXT PREPROCESSING

**Assignment 03 — Neural Networks and Representation Learning**

**Bài toán:** Phân loại nhị phân — Dự đoán sản phẩm có được khuyên dùng hay không (`Recommended IND`) dựa trên văn bản đánh giá (`Review Text`).
**Dữ liệu:** Womens Clothing E-Commerce Reviews

---

## 1. Import libraries & Load data

In [4]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import joblib

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_PATH = os.path.join('..', 'data', 'Womens Clothing E-Commerce Reviews.csv')
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=['Unnamed: 0', 'Clothing ID', 'Title'])
print('Raw shape:', df.shape)
df.head(3)

Raw shape: (23486, 8)


,Age,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,33,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,34,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,60,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses


## 2. Tiền xử lý dữ liệu (Text Cleaning)

In [5]:
# Xóa các hàng mất Review Text (feature chính)
df = df.dropna(subset=['Review Text', 'Recommended IND']).copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9 ]+', '', text) # Chỉ giữ lại chữ và số
    text = re.sub(r'\s+', ' ', text).strip() # Xóa khoảng trắng thừa
    return text

df['Clean_Review'] = df['Review Text'].apply(clean_text)

# Lọc những đánh giá quá ngắn
df['Review_Length'] = df['Clean_Review'].apply(lambda x: len(x.split()))
df = df[df['Review_Length'] > 3] # Chứa ít nhất 3 từ

print('Sau khi clean text:', df.shape)
df[['Review Text', 'Clean_Review', 'Recommended IND']].head()

Sau khi clean text: (22622, 10)


,Review Text,Clean_Review,Recommended IND
0,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky and sexy and comfor...,1
1,Love this dress! it's sooo pretty. i happene...,love this dress its sooo pretty i happened to ...,1
2,I had such high hopes for this dress and reall...,i had such high hopes for this dress and reall...,0
3,"I love, love, love this jumpsuit. it's fun, fl...",i love love love this jumpsuit its fun flirty ...,1
4,This shirt is very flattering to all due to th...,this shirt is very flattering to all due to th...,1


## 3. Khám phá (EDA)

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.countplot(data=df, x='Recommended IND', ax=axes[0])
axes[0].set_title('Phân bố Recommended IND')

sns.histplot(df['Review_Length'], bins=50, kde=True, ax=axes[1])
axes[1].set_title('Phân Phối Chiều Dài Đánh Giá (Số Từ)')
plt.show()

### Nhận xét EDA
- Biểu đồ phân bố nhãn cho biết mức độ cân bằng giữa hai lớp `Recommended IND`; nếu một lớp chiếm ưu thế rõ rệt, Accuracy đơn thuần sẽ chưa phản ánh đầy đủ chất lượng mô hình.
- Phân phối độ dài review cho thấy phần lớn đánh giá có độ dài vừa phải, trong khi các review rất ngắn đã được loại bỏ để giảm nhiễu.

## 4. Feature Engineering: Vector hóa văn bản bằng TF-IDF

Mô hình Học máy cổ điển cần dữ liệu số, ta dùng TF-IDF để vectơ hóa văn bản.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_text = df['Clean_Review']
y = df['Recommended IND'].values

# Chia Train / Val / Test (70/15/15)
X_train_text, X_temp_text, y_train, y_temp = train_test_split(X_text, y, test_size=0.3, random_state=42, stratify=y)
X_val_text, X_test_text, y_val, y_test = train_test_split(X_temp_text, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# TF-IDF Vectorization (sử dụng tối đa 3000 features phổ biến nhất)
tfidf = TfidfVectorizer(max_features=3000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train_text).toarray()
X_val_tfidf = tfidf.transform(X_val_text).toarray()
X_test_tfidf = tfidf.transform(X_test_text).toarray()

print('Train TF-IDF:', X_train_tfidf.shape)
print('Val TF-IDF:', X_val_tfidf.shape)
print('Test TF-IDF:', X_test_tfidf.shape)

Train TF-IDF: (15835, 3000)
Val TF-IDF: (3393, 3000)
Test TF-IDF: (3394, 3000)


### Nhận xét vector hóa
- TF-IDF tạo ra tối đa 3.000 đặc trưng từ vựng và chỉ học từ tập train, giúp tránh rò rỉ thông tin từ validation/test.
- Tỷ lệ chia 70/15/15 và `stratify=y` giữ phân bố nhãn tương đối ổn định giữa ba tập, phù hợp để so sánh các mô hình.

## 5. Lưu dữ liệu đã xử lý

In [8]:
MODEL_DIR = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# Lưu TF-IDF model
joblib.dump(tfidf, os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))

# Lưu Dữ liệu gốc (để Deep Learning PyTorch dùng tokenization / embedding riêng) 
np.savez_compressed(os.path.join(MODEL_DIR, 'raw_text_data.npz'),
                    X_train=X_train_text.values, y_train=y_train,
                    X_val=X_val_text.values, y_val=y_val,
                    X_test=X_test_text.values, y_test=y_test)

# Lưu Dữ liệu TF-IDF (dành cho ML truyền thống)
np.savez_compressed(os.path.join(MODEL_DIR, 'tfidf_data.npz'),
                    X_train=X_train_tfidf, y_train=y_train,
                    X_val=X_val_tfidf, y_val=y_val,
                    X_test=X_test_tfidf, y_test=y_test)
print('✅ Đã lưu vectorizer và biến.')

✅ Đã lưu vectorizer và biến.
